# 7. LightGBM

SECOM 라벨을 PASS=0, FAIL=1로 변환해 LightGBM을 학습하고 불량 클래스 성능을 평가합니다.

각 노트북은 학습/테스트 분할부터 독립 실행할 수 있습니다. 모델 전처리는 학습 데이터에만 맞춰 테스트 데이터 누수를 방지합니다.

## 공통 데이터 준비

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

data_dir = Path.cwd() / "data"
if not data_dir.is_dir():
    data_dir = Path.cwd().parent / "data"
if not data_dir.is_dir():
    raise FileNotFoundError("data 폴더를 찾을 수 없습니다. 프로젝트 루트 또는 notebooks 폴더에서 실행하세요.")

X_raw = pd.read_csv(data_dir / "secom.data", sep=r"\s+", header=None)
labels = pd.read_csv(data_dir / "secom_labels.data", sep=r"\s+", header=None)
y = labels.iloc[:, 0].astype(int)

if len(X_raw) != len(y):
    raise ValueError(f"센서 데이터와 라벨 행 수가 다릅니다: {len(X_raw)} != {len(y)}")

# 분할 후 학습 데이터에서만 결측률 기준과 중앙값을 학습해 데이터 누수를 막습니다.
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)
keep_columns = X_train_raw.columns[X_train_raw.isna().mean() < 0.8]
X_train_raw = X_train_raw.loc[:, keep_columns]
X_test_raw = X_test_raw.loc[:, keep_columns]

imputer = SimpleImputer(strategy="median")
X_train = pd.DataFrame(
    imputer.fit_transform(X_train_raw),
    columns=keep_columns,
    index=X_train_raw.index,
)
X_test = pd.DataFrame(
    imputer.transform(X_test_raw),
    columns=keep_columns,
    index=X_test_raw.index,
)

keep_columns = X_train.columns[X_train.nunique(dropna=False) > 1]
X_train = X_train.loc[:, keep_columns]
X_test = X_test.loc[:, keep_columns]

print(f"학습 데이터: {X_train.shape}, 테스트 데이터: {X_test.shape}")
print("학습 라벨 분포:")
print(y_train.value_counts().sort_index())


학습 데이터: (1253, 466), 테스트 데이터: (314, 466)
학습 라벨 분포:
0
-1    1170
 1      83
Name: count, dtype: int64


## 라벨 변환 및 모델 학습

In [2]:
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

TARGET_NAMES = ["PASS (0)", "FAIL (1)"]


def report_binary_results(y_true, y_pred):
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=TARGET_NAMES,
        zero_division=0,
    ))
    print("Confusion matrix (actual rows, predicted columns):")
    print(confusion_matrix(y_true, y_pred, labels=[0, 1]))


y_train_binary = (y_train == 1).astype(int)
y_test_binary = (y_test == 1).astype(int)

lgb_model = LGBMClassifier(
    random_state=42,
    verbosity=-1,
    n_jobs=-1,
)
lgb_model.fit(X_train, y_train_binary)
pred_lgb = lgb_model.predict(X_test)

report_binary_results(y_test_binary, pred_lgb)

Accuracy: 0.9331
              precision    recall  f1-score   support

    PASS (0)       0.93      1.00      0.97       293
    FAIL (1)       0.00      0.00      0.00        21

    accuracy                           0.93       314
   macro avg       0.47      0.50      0.48       314
weighted avg       0.87      0.93      0.90       314

Confusion matrix (actual rows, predicted columns):
[[293   0]
 [ 21   0]]


In [3]:
from pathlib import Path
import pandas as pd

RESULTS_DIR = data_dir.parent / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def normalize_labels(values):
    values = pd.Series(values).reset_index(drop=True)
    unique_values = set(values.dropna().unique())

    if unique_values.issubset({-1, 1}):
        values = values.map({-1: 0, 1: 1})

    return values.astype(int)


MODEL_NAME = "07LightGBM"
y_true_saved = normalize_labels(y_test)
y_pred_saved = normalize_labels(pred_lgb)

result_df = pd.DataFrame({
    "y_true": y_true_saved,
    "y_pred": y_pred_saved,
})

result_path = RESULTS_DIR / f"{MODEL_NAME}.csv"
result_df.to_csv(result_path, index=False)

print(f"저장 완료: {result_path.name}")
print(result_df.head())

저장 완료: 07LightGBM.csv
   y_true  y_pred
0       0       0
1       0       0
2       0       0
3       0       0
4       0       0


In [4]:
from lightgbm import LGBMClassifier
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

TARGET_NAMES = ["PASS (0)", "FAIL (1)"]


def report_binary_results(y_true, y_pred):
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=[0, 1],
            target_names=TARGET_NAMES,
            zero_division=0,
        )
    )
    print("Confusion matrix (actual rows, predicted columns):")
    print(confusion_matrix(y_true, y_pred, labels=[0, 1]))


y_train_binary = (y_train == 1).astype(int)
y_test_binary = (y_test == 1).astype(int)

# 1. 불균형 비율 계산 (약 14.1)
scale_weight = (y_train_binary == 0).sum() / (y_train_binary == 1).sum()
print(f"적용된 scale_pos_weight: {scale_weight:.2f}")

# 2. 과적합 방지 및 센서 피처 샘플링을 적용한 LightGBM 정의
lgb_model = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    num_leaves=15,  # 리프 노드 수를 줄여 과적합 방지
    max_depth=4,  # 깊이 제한
    scale_pos_weight=scale_weight,
    subsample=0.8,
    colsample_bytree=0.5,  # 센서 피처 무작위 50% 샘플링
    random_state=42,
    verbosity=-1,
    n_jobs=-1,
)
lgb_model.fit(X_train, y_train_binary)

# 3. 불량 예측 확률 추출
pred_prob_lgb = lgb_model.predict_proba(X_test)[:, 1]

# 4. 임계값(Threshold) 탐색 (최적 지점 확인용)
threshold_results = []
for th in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
  temp_pred = (pred_prob_lgb >= th).astype(int)
  detected_fail = ((y_test_binary == 1) & (temp_pred == 1)).sum()
  false_alarm = ((y_test_binary == 0) & (temp_pred == 1)).sum()

  threshold_results.append({
      "Threshold": th,
      "FAIL Recall": recall_score(y_test_binary, temp_pred, pos_label=1),
      "FAIL Precision": precision_score(
          y_test_binary, temp_pred, pos_label=1, zero_division=0
      ),
      "FAIL F1": f1_score(
          y_test_binary, temp_pred, pos_label=1, zero_division=0
      ),
      "검출된 불량(총 21개)": f"{detected_fail}개",
      "정상 오탐(총 293개)": f"{false_alarm}개",
  })

th_df = pd.DataFrame(threshold_results)
print("\n=== LightGBM 임계값별 탐색 결과 ===")
display(th_df)

적용된 scale_pos_weight: 14.10



=== LightGBM 임계값별 탐색 결과 ===


,Threshold,FAIL Recall,FAIL Precision,FAIL F1,검출된 불량(총 21개),정상 오탐(총 293개)
0,0.10,0.571429,0.112150,0.187500,12개,95개
1,0.15,0.523810,0.164179,0.250000,11개,56개
2,0.20,0.428571,0.191489,0.264706,9개,38개
3,0.25,0.333333,0.200000,0.250000,7개,28개
4,0.30,0.238095,0.227273,0.232558,5개,17개
5,0.35,0.190476,0.235294,0.210526,4개,13개
6,0.40,0.142857,0.214286,0.171429,3개,11개
7,0.50,0.095238,0.200000,0.129032,2개,8개


In [5]:
from pathlib import Path
import pandas as pd

# F1 최고는 0.20이지만 불량을 2개 더 잡는 0.15 선택 (테스트셋 기준이라 낙관적, 08 노트북에서 CV로 재검증)
selected_threshold = 0.15
pred_lgb = (pred_prob_lgb >= selected_threshold).astype(int)

print(f"=== 선택된 임계값: {selected_threshold} 적용 결과 ===")
report_binary_results(y_test_binary, pred_lgb)

# 결과 저장
RESULTS_DIR = data_dir.parent / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def normalize_labels(values):
  values = pd.Series(values).reset_index(drop=True)
  unique_values = set(values.dropna().unique())
  if unique_values.issubset({-1, 1}):
    values = values.map({-1: 0, 1: 1})
  return values.astype(int)


# 튜닝 모델 파일명을 LightGBM1로 지정
MODEL_NAME = "07LightGBM1"
y_true_saved = normalize_labels(y_test)
y_pred_saved = normalize_labels(pred_lgb)

result_df = pd.DataFrame({
    "y_true": y_true_saved,
    "y_pred": y_pred_saved,
})

result_path = RESULTS_DIR / f"{MODEL_NAME}.csv"
result_df.to_csv(result_path, index=False)

print(f"\n저장 완료: {result_path.name}")

=== 선택된 임계값: 0.15 적용 결과 ===
Accuracy: 0.7898
              precision    recall  f1-score   support

    PASS (0)       0.96      0.81      0.88       293
    FAIL (1)       0.16      0.52      0.25        21

    accuracy                           0.79       314
   macro avg       0.56      0.67      0.56       314
weighted avg       0.91      0.79      0.84       314

Confusion matrix (actual rows, predicted columns):
[[237  56]
 [ 10  11]]

저장 완료: 07LightGBM1.csv
